# Acoustic Pressure Field Training Data Generator
## k-Wave Simulation for Gravity-Like Ultrasound Force Fields

This notebook generates k-Wave simulation training data for acoustic pressure fields in two geometries:
1. **12-Well Plate** — single well (cylindrical cross-section, ~22 mm diameter)
2. **Flat Microscope Slide** — thin rectangular liquid layer

**Goal:** Simulate multiple transducer configurations that approximate a *"gravity-like"* (uniform downward) acoustic radiation force field, and save pressure field snapshots as training data for the Underdamped Langevin Inference (ULI) algorithm.

**Output:** HDF5 files containing pressure fields, RMS pressure, acoustic intensity, and radiation force density maps — ready to feed into the ULI pipeline.

---
### References
- Brückner, Ronceray & Broedersz, *Phys. Rev. Lett.* 125, 058103 (2020) — ULI algorithm  
- Mizrahi et al., *Soft Matter* 8, 2438 (2012) — LIPUS cytoskeletal perturbation  
- Gupta et al., *Ultrasound Med. Biol.* 48(9), 1745 (2022) — cell culture plate geometry considerations  


In [1]:
# Cell 1 — Install Dependencies
# Run this cell once if k-wave-python is not yet installed in your environment.
# !pip install k-wave-python numpy h5py matplotlib tqdm
print("Dependencies ready — uncomment the line above if needed.")


Dependencies ready — uncomment the line above if needed.


## Imports

In [2]:
import numpy as np
import h5py
import json
import os
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Literal

# k-wave-python
from kwave.kgrid import kWaveGrid
from kwave.kmedium import kWaveMedium
from kwave.ksource import kSource
from kwave.ksensor import kSensor
from kwave.kspaceFirstOrder2D import kspaceFirstOrder2D
from kwave.options.simulation_options import SimulationOptions
from kwave.options.simulation_execution_options import SimulationExecutionOptions

print("All imports successful.")


All imports successful.


## Physical Constants & Geometry Definitions

In [3]:
# ── Tissue/culture medium properties (cell culture media ≈ water at 37°C) ──
MEDIUM_SOUND_SPEED = 1540.0   # m/s
MEDIUM_DENSITY     = 1000.0   # kg/m³
MEDIUM_ALPHA_COEFF = 0.002    # dB/(MHz^y · cm), nearly lossless for water

# ── Simulation frequency (1 MHz = standard LIPUS) ──
FREQUENCY = 1.0e6  # Hz

# ── Grid resolution ──
# Points per wavelength: 6–10 is typical. Higher = more accurate but slower.
PPW = 8
DX = MEDIUM_SOUND_SPEED / (PPW * FREQUENCY)  # ~0.19 mm at 1 MHz, 8 PPW
CFL = 0.3  # Courant–Friedrichs–Lewy number for stable time stepping

# ── 12-well plate geometry (single well) ──
# Standard inner diameter ~22.1 mm, typical seeding depth ~7 mm
WELL_DIAMETER_M = 0.0221
WELL_DEPTH_M    = 0.007

# ── Microscope slide geometry (region of interest) ──
SLIDE_WIDTH_M = 0.025   # 25 mm ROI
SLIDE_DEPTH_M = 0.003   # 3 mm thin liquid layer

# ── Absorbing boundary (PML) thickness in grid points ──
PML_SIZE = 20

print(f"Grid spacing:  {DX*1e3:.3f} mm  ({PPW} PPW at {FREQUENCY/1e6:.1f} MHz)")
print(f"Well grid:     ~{int(WELL_DIAMETER_M/DX)} × {int(WELL_DEPTH_M/DX)} points")
print(f"Slide grid:    ~{int(SLIDE_WIDTH_M/DX)} × {int(SLIDE_DEPTH_M/DX)} points")


Grid spacing:  0.192 mm  (8 PPW at 1.0 MHz)
Well grid:     ~114 × 36 points
Slide grid:    ~129 × 15 points


## Data Classes

In [4]:
@dataclass
class TransducerConfig:
    """Defines a single transducer element: position, phase, and amplitude."""
    x_norm:    float         # Normalized horizontal position (0 = left wall, 1 = right wall)
    phase_deg: float = 0.0   # Phase offset in degrees
    amplitude: float = 1.0   # Relative amplitude (1.0 = full)


@dataclass
class SimulationParams:
    """All parameters for one simulation run (used for logging/metadata)."""
    geometry:           Literal["well", "slide"]
    transducers:        list
    frequency_hz:       float
    ppw:                int
    cfl:                float
    medium_sound_speed: float
    medium_density:     float
    sim_index:          int


## Grid Construction

In [5]:
def make_grid(geometry: str):
    """
    Create kWaveGrid for the chosen geometry.
    
    Convention:
      x = horizontal axis
      y = vertical / depth axis  (y=0 = top, y=Ny = bottom / cell layer)
    Transducers fire from the TOP; cells sit at the BOTTOM.

    Returns (kgrid, (Nx, Ny)).
    """
    if geometry == "well":
        width_m, depth_m = WELL_DIAMETER_M, WELL_DEPTH_M
    elif geometry == "slide":
        width_m, depth_m = SLIDE_WIDTH_M, SLIDE_DEPTH_M
    else:
        raise ValueError(f"Unknown geometry: {geometry!r}. Choose 'well' or 'slide'.")

    Nx = int(np.ceil(width_m / DX))
    Ny = int(np.ceil(depth_m / DX))

    # k-Wave requires even grid dimensions
    Nx += Nx % 2
    Ny += Ny % 2

    kgrid = kWaveGrid([Nx, Ny], [DX, DX])
    print(f"  Grid ({geometry}): {Nx} × {Ny} points  |  "
          f"{Nx*DX*1e3:.1f} mm × {Ny*DX*1e3:.1f} mm")
    return kgrid, (Nx, Ny)


def make_medium():
    """Homogeneous aqueous medium representing cell culture media."""
    return kWaveMedium(
        sound_speed=MEDIUM_SOUND_SPEED,
        density=MEDIUM_DENSITY,
        alpha_coeff=MEDIUM_ALPHA_COEFF,
        alpha_power=2.0,
    )


## Transducer Source Definition

In [6]:
def place_transducers(kgrid, Nx: int, Ny: int, configs: list[TransducerConfig]):
    """
    Place transducer point sources along the TOP wall (y = PML_SIZE + 1).
    Each transducer is a point source with an independent phase and amplitude.

    For a real finite-aperture array, replace the single-point mask entry
    with a line segment (use make_rect from kwave.utils.mapgen).

    Returns (kSource, dt, n_timesteps).
    """
    source = kSource()
    y_src = PML_SIZE + 1  # Just inside the absorbing boundary at the top

    source_mask = np.zeros((Nx, Ny), dtype=bool)
    source_entries = []

    for cfg in configs:
        x_idx = int(np.clip(cfg.x_norm * Nx, 1, Nx - 2))
        source_mask[x_idx, y_src] = True
        source_entries.append((x_idx, cfg.phase_deg, cfg.amplitude))

    source.p_mask = source_mask.astype(int)

    # Build tone-burst signals for each source point
    t_end = 5.0 / FREQUENCY          # 5 acoustic cycles
    dt    = CFL * DX / MEDIUM_SOUND_SPEED
    t_array = np.arange(0, t_end, dt)
    n_t = len(t_array)

    num_sources = int(source_mask.sum())
    source_signals = np.zeros((num_sources, n_t))

    # Flat (column-major) indices of active source points
    active_flat = np.where(source_mask.flatten(order='F'))[0]

    for pos, (x_idx, phase_deg, amplitude) in enumerate(source_entries):
        flat_idx = np.ravel_multi_index([x_idx, y_src], (Nx, Ny), order='F')
        src_pos  = np.where(active_flat == flat_idx)[0]
        if len(src_pos) == 0:
            src_pos = [pos % num_sources]
        phase_rad = np.deg2rad(phase_deg)
        signal    = amplitude * np.sin(2 * np.pi * FREQUENCY * t_array + phase_rad)
        taper     = np.hanning(n_t)           # Hanning window to reduce spectral leakage
        source_signals[src_pos[0], :] = signal * taper

    source.p = source_signals
    return source, dt, n_t


## Sensor Definition

In [7]:
def make_sensor(Nx: int, Ny: int):
    """
    Record pressure across the full 2-D domain.
    p_final  → snapshot at the last timestep (structure of the field)
    p        → full time series at every grid point (for RMS computation)
    """
    sensor = kSensor()
    sensor.mask   = np.ones((Nx, Ny), dtype=int)
    sensor.record = ['p', 'p_final']
    return sensor


## Transducer Configuration Generator

In [8]:
def build_gravity_like_configs(geometry: str, n_transducers_list: list[int]):
    """
    Generate transducer array configurations designed to produce a net
    downward acoustic radiation force — analogous to a uniform gravity field.

    For each count N, three variants are produced:
      1. Uniform in-phase     — pure downward, equal weights
      2. Gaussian apodization — reduced sidelobe / edge diffraction artifacts
      3. Small beam steering  — 5° tilt to test near-vertical sensitivity

    Returns a list of lists of TransducerConfig (one list per configuration).
    """
    width_m    = WELL_DIAMETER_M if geometry == "well" else SLIDE_WIDTH_M
    wavelength = MEDIUM_SOUND_SPEED / FREQUENCY
    all_configs = []

    for n in n_transducers_list:
        if n == 1:
            all_configs.append([TransducerConfig(x_norm=0.5, phase_deg=0.0)])
            continue

        positions = np.linspace(0.1, 0.9, n)

        # 1 — Uniform in-phase
        all_configs.append([TransducerConfig(x_norm=float(p)) for p in positions])

        # 2 — Gaussian amplitude taper
        weights = np.exp(-0.5 * ((positions - 0.5) / 0.25) ** 2)
        all_configs.append([
            TransducerConfig(x_norm=float(p), amplitude=float(w))
            for p, w in zip(positions, weights)
        ])

        # 3 — Linear phase gradient (5° beam steering)
        d          = (0.9 - 0.1) / (n - 1) * width_m   # inter-element spacing
        delta_phi  = 360.0 * d * np.sin(np.deg2rad(5.0)) / wavelength
        phases     = np.arange(n) * delta_phi
        all_configs.append([
            TransducerConfig(x_norm=float(p), phase_deg=float(ph))
            for p, ph in zip(positions, phases)
        ])

    return all_configs


## Single Simulation Runner

In [12]:
def run_simulation(geometry: str, configs: list[TransducerConfig],
                   sim_idx: int, output_dir: str):
    """
    Execute one k-Wave simulation and save results to an HDF5 file.

    Saved datasets per file:
      p_final              — pressure snapshot at final timestep  (Nx, Ny)
      p_rms                — RMS pressure over the simulation     (Nx, Ny)
      intensity            — acoustic intensity I = p_rms²/(ρc)  (Nx, Ny)  [W/m²]
      radiation_force_dens — F = 2α·I/c  (simplified body force) (Nx, Ny)  [N/m³]
      transducer_x_norm    — normalized x positions of transducers
      transducer_phase_deg — phase offsets
      transducer_amplitude — relative amplitudes

    Returns (file_path, metadata_dict).
    """
    kgrid, (Nx, Ny) = make_grid(geometry)
    medium = make_medium()
    source, dt, n_t = place_transducers(kgrid, Nx, Ny, configs)
    sensor = make_sensor(Nx, Ny)

    kgrid.setTime(n_t, dt)

    sim_options  = SimulationOptions(
        save_to_disk=True, 
        input_filename='kwave_input.h5',
        output_filename='kwave_output.h5',
        pml_size=PML_SIZE, pml_inside=False,
    )
    exec_options = SimulationExecutionOptions(is_gpu_simulation=False)

    print(f"  [{sim_idx:04d}] {geometry:5s} | {Nx}×{Ny} grid | "
          f"{len(configs)} transducer(s) | {n_t} timesteps")

    sensor_data = kspaceFirstOrder2D(
        kgrid, source, sensor, medium,
        simulation_options=sim_options,
        execution_options=exec_options,
    )

    p_final = sensor_data['p_final']                          # (Nx, Ny)
    p_rms   = np.sqrt(
        np.mean(sensor_data['p'].reshape(Nx, Ny, -1) ** 2, axis=-1)
    )                                                          # (Nx, Ny)

    intensity = p_rms ** 2 / (MEDIUM_DENSITY * MEDIUM_SOUND_SPEED)

    # Attenuation in Np/m (simplified; valid for low-loss aqueous media)
    alpha_np_m = MEDIUM_ALPHA_COEFF * 100 / (8.686 * FREQUENCY / 1e6)
    radiation_force_density = 2 * alpha_np_m * intensity / MEDIUM_SOUND_SPEED

    fname = os.path.join(output_dir, f"sim_{sim_idx:04d}.h5")
    with h5py.File(fname, 'w') as f:
        f.create_dataset('p_final',              data=p_final.astype(np.float32))
        f.create_dataset('p_rms',                data=p_rms.astype(np.float32))
        f.create_dataset('intensity',            data=intensity.astype(np.float32))
        f.create_dataset('radiation_force_dens', data=radiation_force_density.astype(np.float32))
        f.attrs['geometry']      = geometry
        f.attrs['Nx']            = Nx
        f.attrs['Ny']            = Ny
        f.attrs['dx_m']          = DX
        f.attrs['frequency_hz']  = FREQUENCY
        f.attrs['n_transducers'] = len(configs)
        f.create_dataset('transducer_x_norm',    data=np.array([c.x_norm    for c in configs]))
        f.create_dataset('transducer_phase_deg', data=np.array([c.phase_deg for c in configs]))
        f.create_dataset('transducer_amplitude', data=np.array([c.amplitude for c in configs]))

    meta = {
        'sim_index': sim_idx, 'geometry': geometry,
        'n_transducers': len(configs),
        'transducers': [asdict(c) for c in configs],
        'Nx': Nx, 'Ny': Ny, 'dx_m': DX, 'file': fname,
    }
    return fname, meta


## Training Dataset Generator

In [13]:
def generate_training_dataset(
    geometries: list[str]          = ['well', 'slide'],
    n_transducers_list: list[int]  = [1, 2, 4, 8],
    output_root: str               = 'training_data',
):
    """
    Main loop: iterate over geometries and transducer configurations,
    run simulations, and save results + a metadata JSON index file.
    """
    Path(output_root).mkdir(exist_ok=True)
    all_metadata = []
    sim_idx = 0

    for geometry in geometries:
        geo_dir = os.path.join(output_root, geometry)
        Path(geo_dir).mkdir(exist_ok=True)

        configs_list = build_gravity_like_configs(geometry, n_transducers_list)
        print(f"\n{'='*60}")
        print(f"Geometry: {geometry.upper()}  |  {len(configs_list)} configurations")
        print(f"{'='*60}")

        for configs in configs_list:
            _, meta = run_simulation(geometry, configs, sim_idx, geo_dir)
            all_metadata.append(meta)
            sim_idx += 1

    meta_path = os.path.join(output_root, 'metadata.json')
    with open(meta_path, 'w') as f:
        json.dump(all_metadata, f, indent=2)

    print(f"\n✓ Completed {sim_idx} simulations. Metadata → {meta_path}")
    return all_metadata


## ▶ Run Simulations

In [14]:
# ── Adjust these parameters to control the sweep ──────────────────────────
GEOMETRIES         = ['well', 'slide']   # 'well' = 12-well plate, 'slide' = microscope slide
N_TRANSDUCERS_LIST = [1, 2, 4, 8]       # number of transducer elements per config
OUTPUT_ROOT        = 'training_data'    # output directory

# ── Run ───────────────────────────────────────────────────────────────────
metadata = generate_training_dataset(
    geometries=GEOMETRIES,
    n_transducers_list=N_TRANSDUCERS_LIST,
    output_root=OUTPUT_ROOT,
)



Geometry: WELL  |  10 configurations
  Grid (well): 116 × 38 points  |  22.3 mm × 7.3 mm
  [0000] well  | 116×38 grid | 1 transducer(s) | 134 timesteps



dyld[20215]: Library not loaded: /opt/homebrew/opt/fftw/lib/libfftw3f.3.dylib
  Referenced from: <5B82F0C4-5B35-3AFA-A687-C07AC713836E> /Users/davisbone/Repositories/Bespoke-Ultrasound/.venv/lib/python3.9/site-packages/kwave/bin/darwin/kspaceFirstOrder-OMP
  Reason: tried: '/opt/homebrew/opt/fftw/lib/libfftw3f.3.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/fftw/lib/libfftw3f.3.dylib' (no such file), '/opt/homebrew/opt/fftw/lib/libfftw3f.3.dylib' (no such file)



CalledProcessError: Command '['/Users/davisbone/Repositories/Bespoke-Ultrasound/.venv/lib/python3.9/site-packages/kwave/bin/darwin/kspaceFirstOrder-OMP', '-i', '/var/folders/tz/dtb_3rdj0j73z9pzh5q2bm5h0000gn/T/kwave_input.h5', '-o', '/var/folders/tz/dtb_3rdj0j73z9pzh5q2bm5h0000gn/T/kwave_output.h5', ' --p_raw --p_final -s 1']' died with <Signals.SIGABRT: 6>.

## Visualization

In [13]:
import matplotlib.pyplot as plt

def plot_pressure_field(h5_path: str, save_png: bool = True):
    """
    Load a saved simulation file and display:
      Left  — RMS Pressure field (Pa)
      Right — Radiation Force Density (N/m³)
    Transducer positions are marked with dashed cyan lines.
    """
    with h5py.File(h5_path, 'r') as f:
        p_rms   = f['p_rms'][:]
        rf_dens = f['radiation_force_dens'][:]
        Nx      = f.attrs['Nx']
        Ny      = f.attrs['Ny']
        dx      = f.attrs['dx_m']
        geo     = f.attrs['geometry']
        n_tx    = f.attrs['n_transducers']
        x_norms = f['transducer_x_norm'][:]

    x_mm = np.arange(Nx) * dx * 1e3
    y_mm = np.arange(Ny) * dx * 1e3

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"Geometry: {geo}  |  {n_tx} transducer(s)", fontsize=13, fontweight='bold')

    kw = dict(origin='upper', aspect='auto',
              extent=[x_mm[0], x_mm[-1], y_mm[-1], y_mm[0]])

    im0 = axes[0].imshow(p_rms.T, cmap='inferno', **kw)
    axes[0].set_title('RMS Pressure (Pa)')
    axes[0].set_xlabel('x (mm)')
    axes[0].set_ylabel('Depth / y (mm)  [↓ gravity direction]')
    plt.colorbar(im0, ax=axes[0])
    for xn in x_norms:
        axes[0].axvline(x=xn * x_mm[-1], color='cyan', lw=1.2, ls='--', alpha=0.75,
                        label='Transducer')

    im1 = axes[1].imshow(rf_dens.T, cmap='viridis', **kw)
    axes[1].set_title('Radiation Force Density (N/m³)')
    axes[1].set_xlabel('x (mm)')
    axes[1].set_ylabel('Depth / y (mm)')
    plt.colorbar(im1, ax=axes[1])

    plt.tight_layout()
    if save_png:
        out = h5_path.replace('.h5', '_preview.png')
        plt.savefig(out, dpi=150)
        print(f"Preview saved → {out}")
    plt.show()


# Plot the first result
if metadata:
    plot_pressure_field(metadata[0]['file'])


NameError: name 'metadata' is not defined

## PyTorch Dataset Class (for ULI Training)

In [ ]:
class AcousticFieldDataset:
    """
    Lazy-loading PyTorch-compatible dataset over the HDF5 simulation files.

    Each item returns:
      field  — (1, Nx, Ny) float32 tensor, normalized to [0, 1]
      label  — (24,) float32 vector encoding transducer positions,
               phases (normalized to [0,1]), and amplitudes
               for up to 8 transducer elements (padded with zeros)

    Usage:
        from torch.utils.data import DataLoader
        ds     = AcousticFieldDataset('training_data/metadata.json', field='p_rms')
        loader = DataLoader(ds, batch_size=8, shuffle=True)
        for fields, labels in loader:
            # fields: (B, 1, Nx, Ny)  labels: (B, 24)
            ...
    """

    MAX_TX = 8  # label vector supports up to 8 transducers (pad with zeros if fewer)

    def __init__(self, metadata_json: str, field: str = 'p_rms',
                 geometry_filter: str | None = None):
        with open(metadata_json) as f:
            self.meta = json.load(f)
        if geometry_filter:
            self.meta = [m for m in self.meta if m['geometry'] == geometry_filter]
        self.field = field

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        m = self.meta[idx]
        with h5py.File(m['file'], 'r') as f:
            field_data = f[self.field][:]
            x_norms    = f['transducer_x_norm'][:]
            phases     = f['transducer_phase_deg'][:]
            amps       = f['transducer_amplitude'][:]

        # Normalize field to [0, 1]
        mn, mx = field_data.min(), field_data.max()
        field_data = (field_data - mn) / (mx - mn + 1e-9)

        # Build fixed-length label vector [positions | phases | amplitudes]
        label = np.zeros(self.MAX_TX * 3, dtype=np.float32)
        n = min(len(x_norms), self.MAX_TX)
        label[:n]                          = x_norms[:n]
        label[self.MAX_TX:self.MAX_TX+n]   = phases[:n] / 360.0
        label[2*self.MAX_TX:2*self.MAX_TX+n] = amps[:n]

        return field_data.astype(np.float32)[np.newaxis, ...], label

# Quick sanity check (no PyTorch required)
if metadata:
    ds = AcousticFieldDataset('training_data/metadata.json')
    field, label = ds[0]
    print(f"Dataset length : {len(ds)}")
    print(f"Field shape    : {field.shape}")
    print(f"Label shape    : {label.shape}")
    print(f"Field range    : [{field.min():.3f}, {field.max():.3f}]")
    print(f"Label (first 8 positions): {label[:8]}")
